# 02 — Predição, desempenho e explicabilidade em classificação

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/02_aprendizado_supervisionado.ipynb)

**Duração estimada:** 75–90 minutos  
**Pré-requisitos:** Notebook 01 ou noções de pandas e classificação binária.

## Objetivos

- construir um pipeline sem vazamento
- usar cinco folds e um teste final intacto
- interpretar sensibilidade, especificidade, ROC-AUC e PR-AUC
- investigar o modelo com permutação e SHAP sem alegar causalidade

## Fonte e licença

[CDC Diabetes Health Indicators — descrição das variáveis](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators), conjunto 891 da UCI, derivado do BRFSS. Consulte a tabela de variáveis para interpretar os códigos dos atributos, como 0 e 1.

Fonte UCI sob CC BY 4.0; cite o conjunto e sua publicação.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como fixar dependências e semente?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1', 'shap': 'shap>=0.45,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.data_loading import load_cdc_diabetes
from src.evaluation import classification_report_health, plot_classification_curves, report_frame

In [ ]:
FAST_MODE = True
SAMPLE_SIZE = 30_000 if FAST_MODE else 40_000
data, metadata = load_cdc_diabetes(SAMPLE_SIZE, random_state=RANDOM_STATE)
X = data.drop(columns="Diabetes_binary")
y = data["Diabetes_binary"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Treino:", X_train.shape, "| teste reservado:", X_test.shape)

## Pergunta orientadora

> Um modelo consegue ordenar pessoas com e sem o indicador-alvo, e quais erros aparecem quando escolhemos um limiar?

## Inspeção de correlações

> Quais associações monotônicas merecem atenção antes do modelo?

In [ ]:
corr = data.corr(method="spearman", numeric_only=True)
target_corr = corr["Diabetes_binary"].drop("Diabetes_binary")
top_corr = target_corr.abs().sort_values(ascending=False).head(12).index
display(target_corr.loc[top_corr].sort_values(key=abs, ascending=False).rename("Spearman").to_frame())
plt.figure(figsize=(10, 7))
sns.heatmap(corr.loc[[*top_corr, "Diabetes_binary"], [*top_corr, "Diabetes_binary"]], cmap="vlag", center=0)
plt.title("Correlação de Spearman entre os principais atributos")
plt.show()

### Como interpretar

Spearman resume associação monotônica. Códigos ordinais, não linearidade e variáveis omitidas afetam a leitura; correlação não demonstra causalidade.

## Preparação e validação

> Como ajustar escala apenas dentro de cada treino?

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["roc_auc", "f1", "recall", "precision", "balanced_accuracy"]
cv_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
display(pd.DataFrame({m: [cv_scores[f"test_{m}"].mean(), cv_scores[f"test_{m}"].std(ddof=1)] for m in scoring}, index=["média", "desvio-padrão"]).T.round(3))

## Avaliação no teste final

> O que acontece no conjunto que não participou da validação?

Para positivos (VP), negativos (VN), falsos positivos (FP) e falsos negativos (FN):

- sensibilidade = VP / (VP + FN);
- especificidade = VN / (VN + FP);
- precisão = VP / (VP + FP);
- F1 = 2 × precisão × sensibilidade / (precisão + sensibilidade).

In [ ]:
pipeline.fit(X_train, y_train)
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
report = classification_report_health(y_test, y_pred, y_prob)
display(report_frame(report).round(3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Sem indicação", "Com indicação"], cmap="Blues")
plt.title("Matriz de confusão — teste")
plt.show()
plot_classification_curves(y_test, y_prob); plt.show()

### Como interpretar

Um falso negativo é um positivo no conjunto que o limiar não sinalizou; um falso positivo é um negativo sinalizado. O custo de cada erro depende do contexto — não é decidido pela ROC-AUC. Alterar 0,5 muda esse equilíbrio.

## Explicabilidade

> Quais atributos mudam mais o desempenho ou a saída de um segundo modelo?

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_i = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_i = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)
forest = RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
forest.fit(X_train_i, y_train)
importance = permutation_importance(forest, X_test_i, y_test, scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
permutation = pd.Series(importance.importances_mean, index=X.columns).nlargest(12)
permutation.sort_values().plot.barh(title="Importância por permutação no teste (ROC-AUC)")
plt.xlabel("Queda média na ROC-AUC"); plt.show()

In [ ]:
import shap

shap_sample = X_test_i.sample(min(300, len(X_test_i)), random_state=RANDOM_STATE)
explainer = shap.Explainer(forest, X_train_i.sample(min(500, len(X_train_i)), random_state=RANDOM_STATE))
shap_values = explainer(shap_sample)
if shap_values.values.ndim == 3:  # saída por classe em versões recentes
    shap_values = shap_values[..., 1]
shap.plots.bar(shap_values, max_display=12)
shap.plots.beeswarm(shap_values, max_display=12)

In [ ]:
main_feature = permutation.index[0]
shap.plots.scatter(shap_values[:, main_feature])
forest_prob = forest.predict_proba(X_test_i)[:, 1]
forest_pred = (forest_prob >= 0.5).astype(int)
examples = pd.DataFrame({"real": y_test.to_numpy(), "predito": forest_pred, "probabilidade": forest_prob}, index=X_test.index)
display(pd.concat([
    examples[examples.real == examples.predito].head(1).assign(tipo="correta"),
    examples[examples.real != examples.predito].head(1).assign(tipo="incorreta"),
]))

### Como interpretar

Permutação mede perda de desempenho ao embaralhar um atributo; SHAP decompõe saídas do modelo. Ambos descrevem o **modelo nesta amostra**. Eles não dizem que mudar o atributo causará mudança de saúde.

## Limitações e responsabilidade

- O alvo e os atributos incluem autorrelato e não representam um diagnóstico produzido pelo notebook.
- Desbalanceamento, limiar, subgrupos e mudança de população alteram os erros.
- SHAP pode refletir correlações, vieses e atalhos do modelo; não é explicação causal.

## Atividade

Teste limiares 0,3; 0,5; 0,7. Compare sensibilidade e especificidade e justifique qual erro seria prioritário em um contexto hipotético — sem fazer recomendação clínica.

## Três aprendizados principais

1. Pipeline e folds impedem que o pré-processamento enxergue a validação.
2. ROC-AUC não substitui a matriz de confusão nem a PR-AUC.
3. Explicar uma previsão não equivale a descobrir sua causa.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [SHAP documentation](https://shap.readthedocs.io/)
- [scikit-learn — model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'shap', 'ucimlrepo'))